# LSTM

Pytorch documentation [here](https://www.pytorch.org/) \
Keras documentation [here](https://keras.io/)

In [ ]:
import os

# This guide can only be run with the torch backend.
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.utils import to_categorical
from keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Load data - California Housing

In [ ]:

# ================================
# 2️⃣ Load the datset
# ================================
data = _
print(data.head())

# ================================
# 3️⃣ Pick variables and target
# ================================
X = data.drop("median_house_value", axis=1).values
y = data["median_house_value"].values


In [ ]:

# ================================
# 4️⃣ Standardize variables
# ================================
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))


In [ ]:
# ================================
# 5️⃣ Reshape data for input LSTM
# ================================
# Convert each row (8 features) in a sequence of 8 features (samples, timesteps, features)
X_seq = X_scaled.reshape(X_scaled.shape[0], X_scaled.shape[1], 1)

In [ ]:
# ================================
# 6️⃣ Holdout (train / test)
# ================================
X_train, X_test, y_train, y_test = train_test_split(X_seq, y_scaled, test_size=0.2, random_state=42)

## Create a graph model

In [ ]:
# ================================
# 7️⃣ LSTM
# ================================
model = Sequential()
model.add(LSTM(64, activation='tanh', input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(32, activation='tanh'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))  # Regression

In [ ]:
model.summary()

## Define loss function and optimizer

In [ ]:
# ================================
# 8️⃣ Define loss, optimizer and metrics
# ================================
model.compile(loss = "mse" , optimizer = "adam" , metrics = ["mae"] )

## Train model

In [ ]:
# ================================
# 9️⃣ Training
# ================================

history = model.fit( X_train , y_train , \
                    #X_test, y_test, \
                    epochs = 2 , batch_size = 64 , \
                    verbose = 1, validation_split = 0.1 )

## Plot results

In [ ]:
# ================================
# 🔟 Plot progress and results
# ================================

_, axes = plt.subplots(1,2, figsize=(10,5))
axes[0].plot(history.history['loss'], c='b')
axes[0].plot(history.history['val_loss'], c='r')
axes[1].plot(history.history['mae'], c='b')
axes[1].plot(history.history['val_mae'], c='r')
axes[0].set_title("Loss"), axes[1].set_title('MAE')

## Compute metrics over ```X_test``` images

In [ ]:
# ================================
# 1️⃣1️⃣ Generate predictions
# ================================

y_pred = model.predict( X_test )

In [ ]:
y_pred_rescaled = scaler_y.inverse_transform(y_pred)
print("\Predictions (Real values):")
print(y_pred_rescaled)

In [ ]:
_, ax = plt.subplots(figsize=(6,5))
ax.plot(y_test, c='b')
ax.plot(y_pred, c='r')
ax.set_title('MAE')
ax.set_xlim(0, 50)